In [0]:
TABLE_PAYMENT_BRONZE = "customer_360.bronze.payments"
TABLE_SILVER_PAYMENT = "customer_360.silver.payments"
TABLE_QUARANTINE_PAYMENT = "customer_360.quarantine.payments"

TABLE_PAYMENT_VIEWS = "customer_360.bronze.payment_views"

PAYMENT_METRICS_TABLE = "customer_360.raw.payment_silver_metrics"

PATH_PAYMENT_CHECKPOINTLOCATION_SILVER = (
    "/Volumes/customer_360/raw/source_files/checkpoints/silver/payments"
)

TABLE_METRIC = "customer_360.raw.stream_metrics"

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS customer_360.silver
""")

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS customer_360.quarantine
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_SILVER_PAYMENT} (
    payment_id STRING NOT NULL,
    order_id STRING NOT NULL,
    customer_id STRING NOT NULL,
    payment_method STRING,
    payment_status STRING,
    amount DECIMAL(12,2),
    payment_date TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_QUARANTINE_PAYMENT} (
    payment_id STRING,
    order_id STRING,
    customer_id STRING,
    payment_method STRING,
    payment_status STRING,
    amount DECIMAL(12,2),
    payment_date TIMESTAMP,
    updated_at TIMESTAMP,
    failure_reason STRING,
    quarantined_at TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_PAYMENT_VIEWS} (
    payment_id STRING NOT NULL,
    updated_at TIMESTAMP,
    viewed_at TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {PAYMENT_METRICS_TABLE} (
    metric_time TIMESTAMP,
    batch_id BIGINT,
    query_name STRING,
    total_records BIGINT,
    valid_records BIGINT,
    invalid_records BIGINT,
    duplicate_records BIGINT
)
USING DELTA
""")

In [0]:
payment_df = (
    spark
    .readStream
    .format("delta")
    .table(TABLE_PAYMENT_BRONZE)
)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from datetime import datetime


def process_payment_dataframe(batch_df, batch_id):

   
    # Standardization + Data Quality Checks
   

    df = (
        batch_df
        .withColumn("payment_id", trim(col("payment_id")))
        .withColumn("order_id", trim(col("order_id")))
        .withColumn("customer_id", trim(col("customer_id")))
        .withColumn("payment_method", trim(col("payment_method")))
        .withColumn("payment_status", trim(col("payment_status")))
        .withColumn(
            "failure_reason",

            when(
                col("payment_id").isNull(),
                "payment_id is null"
            )

            .when(
                col("order_id").isNull(),
                "order_id is null"
            )

            .when(
                col("customer_id").isNull(),
                "customer_id is null"
            )

            .when(
                col("payment_method").isNull(),
                "payment_method is null"
            )

            .when(
                ~col("payment_method").isin(
                    "CREDIT_CARD",
                    "DEBIT_CARD",
                    "UPI",
                    "NET_BANKING",
                    "WALLET",
                    "COD",
                    "CARD"
                ),
                "invalid payment_method"
            )

            .when(
                col("payment_status").isNull(),
                "payment_status is null"
            )

            .when(
                ~col("payment_status").isin(
                    "PENDING",
                    "SUCCESS",
                    "FAILED",
                    "REFUNDED"
                ),
                "invalid payment_status"
            )

            .when(
                col("amount").isNull(),
                "amount is null"
            )

            .when(
                col("amount") <= 0,
                "amount must be greater than 0"
            )

            .when(
                col("payment_date").isNull(),
                "payment_date is null"
            )

            .when(
                col("updated_at").isNull(),
                "updated_at is null"
            )

            .otherwise(None)
        )
    )

   
    # Valid Data
   

    valid_data = (
        df
        .filter(col("failure_reason").isNull())
        .drop("failure_reason")
    )

   
    # Invalid Data → Quarantine
   

    invalid_data = (
        df
        .filter(col("failure_reason").isNotNull())
        .withColumn(
            "quarantined_at",
            current_timestamp()
        )
    )

    invalid_data.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable(TABLE_QUARANTINE_PAYMENT)

   
    # Within-Batch Duplicate Detection
    #
    # Duplicate Key:
    # payment_id + updated_at
   

    window = (
        Window
        .partitionBy(
            ["payment_id", "updated_at"]
        )
        .orderBy(
            col("updated_at").asc()
        )
    )

    valid_data = (
        valid_data
        .withColumn(
            "rn",
            row_number().over(window)
        )
    )

    unique_data = (
        valid_data
        .filter(col("rn") == 1)
        .drop("rn")
    )

    duplicate_data = (
        valid_data
        .filter(col("rn") > 1)
        .drop("rn")
    )

   
    # Load Previously Processed Payments
   

    first_occurance = (
        spark
        .read
        .format("delta")
        .table(TABLE_PAYMENT_VIEWS)
    )

   
    # Previously Seen Payments
   

    seen_data = (
        first_occurance
        .join(
            unique_data,
            on=["payment_id", "updated_at"],
            how="inner"
        )
        .select([
            "payment_id",
            "order_id",
            "customer_id",
            "payment_method",
            "payment_status",
            "amount",
            "payment_date",
            "updated_at"
        ])
    )

   
    # New Payments
   

    silver_df = (
        unique_data
        .join(
            first_occurance,
            on=["payment_id", "updated_at"],
            how="left_anti"
        )
        .select([
            "payment_id",
            "order_id",
            "customer_id",
            "payment_method",
            "payment_status",
            "amount",
            "payment_date",
            "updated_at"
        ])
    )

   
    # Write New Payments → Silver
   

    silver_df.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable(TABLE_SILVER_PAYMENT)

    valid_data_count = silver_df.count()

   
    # Duplicate Data → Quarantine
   

    quarantine_data = (
        duplicate_data
        .unionByName(seen_data)
        .withColumn(
            "failure_reason",
            lit("duplicate record")
        )
        .withColumn(
            "quarantined_at",
            current_timestamp()
        )
    )

    quarantine_data.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable(TABLE_QUARANTINE_PAYMENT)

    duplicate_records = quarantine_data.count()

   
    # Store Processed Payments
   

    viewed_data = (
        silver_df
        .withColumn(
            "viewed_at",
            current_timestamp()
        )
        .select([
            "payment_id",
            "updated_at",
            "viewed_at"
        ])
    )

    viewed_data.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable(TABLE_PAYMENT_VIEWS)

   
    # Silver Batch Metrics
   

    total_records = batch_df.count()

    metric = [
        Row(
            metric_time=datetime.now(),
            batch_id=batch_id,
            query_name="payment_silver",
            total_records=total_records,
            valid_records=valid_data_count,
            invalid_records=total_records - valid_data_count,
            duplicate_records=duplicate_records
        )
    ]

    metric_df = spark.createDataFrame(metric)

    metric_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(PAYMENT_METRICS_TABLE)

In [0]:
query = (
    payment_df
    .writeStream
    .trigger(availableNow=True)
    .foreachBatch(process_payment_dataframe)
    .option(
        "checkpointLocation",
        PATH_PAYMENT_CHECKPOINTLOCATION_SILVER
    )
    .start()
)

query.awaitTermination()

In [0]:
import json
from pyspark.sql import Row
from datetime import datetime

metrics = []

for p in query.recentProgress:

    progress = json.loads(p.json)

    source = progress["sources"][0]

    metrics.append(
        Row(
            metric_time=datetime.now(),
            query_name="payment_silver",
            batch_id=int(progress["batchId"]),
            input_rows=int(
                source.get("numInputRows", 0)
            ),
            input_rows_per_second=float(
                source.get("inputRowsPerSecond", 0.0)
            ),
            processed_rows_per_second=float(
                source.get("processedRowsPerSecond", 0.0)
            ),
            processing_time_ms=int(
                progress
                .get("durationMs", {})
                .get("triggerExecution", 0)
            )
        )
    )

if metrics:

    metrics_df = spark.createDataFrame(metrics)

    metrics_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(TABLE_METRIC)

In [0]:
spark.sql("SELECT * FROM customer_360.bronze.payments").display()

spark.sql("SELECT * FROM customer_360.silver.payments").display()

spark.sql("SELECT * FROM customer_360.bronze.payment_views").display()

spark.sql("SELECT * FROM customer_360.quarantine.payments").display()

spark.sql("SELECT * FROM customer_360.raw.payment_silver_metrics").display()

spark.sql("SELECT * FROM customer_360.raw.stream_metrics").display()